In [1]:
"""
Master dataset = outer join of TMDB, The Numbers and Rotten Tomatoes.

Matching logic (same as the pairwise script):
  1. normalize titles (lowercase, strip accents, keep letters only)
  2. year must be equal
  3. Levenshtein normalized similarity >= THRESHOLD (100 = exact normalized title)

Differences that matter for an outer join:
  - matching is ONE-TO-ONE (greedy, best score first), so a row can never be
    duplicated by being the "best match" of several rows on the other side
  - datasets are joined sequentially: (TMDB outer Numbers) outer Rotten
    -> every movie from every dataset ends up in exactly one master row
"""

import os
import re
import unicodedata

import numpy as np
import pandas as pd
from rapidfuzz import process
from rapidfuzz.distance import Levenshtein

THRESHOLD = 100  # similarity in %, 100 = exact match on normalized title

# Columns dropped before joining (edit freely)
DROP_COLS = {
    "tmdb": ["genre_ids", "original_language"],
    "numbers": ["url"],
    "rotten": [
        "url",
        "release_date_streaming",
        "rerelease date (theaters)",
        "release date (streaming)",
        "box office (gross usa)",
        "release date (tv)",
        "release date (dvd)",
        # "release_date_theaters_month", # ill leave it for seasonnality analysis
        "release_date_theaters_day",
    ],
}


# ------------------------------------------------------------
# Normalization
# ------------------------------------------------------------
def normalize_title(title):
    title = unicodedata.normalize("NFKD", str(title).lower())
    title = "".join(c for c in title if not unicodedata.combining(c))
    title = re.sub(r"[^a-z\s0-9]", "", title) # add numbers back to avoid false positives like "1917" vs "17"
    return re.sub(r"\s+", " ", title).strip()


def prepare(df, name, year_col=None):
    """Normalize title, standardize year, drop junk columns, prefix everything."""
    df = df.copy()
    if year_col and year_col != "year":
        df = df.rename(columns={year_col: "year"})
    df["year"] = pd.to_numeric(df["year"], errors="coerce") # coerce invalid years to NaN
    df["title_normalized"] = df["title"].apply(normalize_title)
    df = df.drop(columns=DROP_COLS.get(name, []), errors="ignore")
    df = df.add_prefix(f"{name}_").reset_index(drop=True) # we reset index to avoid collisions when merging
    return df


# ------------------------------------------------------------
# One-to-one fuzzy matching with year constraint
# ------------------------------------------------------------
def match_one_to_one(l_title, l_year, r_title, r_year, threshold=THRESHOLD):
    """
    Return DataFrame(left_idx, right_idx, score) of one-to-one matches.
    Only pairs with the same year and similarity >= threshold are kept;
    within a year, best scores are assigned first.
    """
    cutoff = threshold / 100 

    l_valid = l_title.ne("") & l_year.notna() # only consider rows with a non-empty title and a valid year
    r_valid = r_title.ne("") & r_year.notna() # only consider rows with a non-empty title and a valid year
    l_groups = l_title[l_valid].groupby(l_year[l_valid]).groups # group by year to avoid cross-year matches
    r_groups = r_title[r_valid].groupby(r_year[r_valid]).groups # group by year to avoid cross-year matches

    rows = []
    for year, l_idx in l_groups.items(): # iterate over each year group in the left dataset
        r_idx = r_groups.get(year) # get the corresponding year group in the right dataset
        if r_idx is None:
            continue

        l_idx = np.asarray(l_idx) # convert to numpy array for indexing
        r_idx = np.asarray(r_idx)

        sims = process.cdist( # compute pairwise similarity scores between titles in the left and right datasets
            l_title.loc[l_idx].tolist(),
            r_title.loc[r_idx].tolist(),
            scorer=Levenshtein.normalized_similarity,
            score_cutoff=cutoff,
            dtype=np.float32,
            workers=-1,
        )

        cand = np.argwhere(sims >= cutoff) # get the indices of pairs with similarity above the threshold
        if len(cand) == 0:
            continue

        scores = sims[cand[:, 0], cand[:, 1]] # get the similarity scores for the candidate pairs
        order = np.argsort(-scores, kind="stable") # sort candidates by descending score, keeping the original order for ties

        used_l, used_r = set(), set() # keep track of which rows have already been matched
        for k in order:
            i, j = cand[k]
            if i in used_l or j in used_r: # if either row has already been matched, skip this pair
                continue
            used_l.add(i)
            used_r.add(j)
            rows.append((l_idx[i], r_idx[j], float(scores[k]) * 100)) # store the matched indices and score (converted to percentage)

    return pd.DataFrame(rows, columns=["left_idx", "right_idx", "score"]) # return a DataFrame containing the matched indices and their similarity scores


def outer_join(left, right, l_title, l_year, r_title, r_year, score_col, threshold=THRESHOLD):
    """Fuzzy outer join: matched rows side by side + unmatched rows from both sides."""
    pairs = match_one_to_one(left[l_title], left[l_year], right[r_title], right[r_year], threshold) # get the one-to-one matches between the two datasets

    matched = pd.concat( # concatenate the matched rows from both datasets side by side
        [
            left.loc[pairs["left_idx"]].reset_index(drop=True),
            right.loc[pairs["right_idx"]].reset_index(drop=True),
        ],
        axis=1,
    )
    matched[score_col] = pairs["score"].round(2).values # add the similarity scores to the matched DataFrame

    left_only = left.drop(index=pairs["left_idx"]) # get the rows from the left dataset that were not matched
    right_only = right.drop(index=pairs["right_idx"])

    return pd.concat([matched, left_only, right_only], ignore_index=True)


# ------------------------------------------------------------
# Diagnostics: ambiguous matches (one row matching several rows)
# ------------------------------------------------------------
def all_candidate_pairs(df1, name1, df2, name2, threshold=THRESHOLD): # this serves as a deduplication check
    """
    EVERY pair (same year, similarity >= threshold), not just the best one.
    df1 / df2 must come from prepare() (prefixed columns, unique index).
    """
    cutoff = threshold / 100 
    t1, y1 = df1[f"{name1}_title_normalized"], df1[f"{name1}_year"] # get the normalized titles and years from both datasets
    t2, y2 = df2[f"{name2}_title_normalized"], df2[f"{name2}_year"]

    v1 = t1.ne("") & y1.notna()
    v2 = t2.ne("") & y2.notna()
    g1 = t1[v1].groupby(y1[v1]).groups
    g2 = t2[v2].groupby(y2[v2]).groups

    rows = []
    for year, idx1 in g1.items():
        idx2 = g2.get(year)
        if idx2 is None:
            continue
        idx1, idx2 = np.asarray(idx1), np.asarray(idx2)
        sims = process.cdist(
            t1.loc[idx1].tolist(), t2.loc[idx2].tolist(),
            scorer=Levenshtein.normalized_similarity,
            score_cutoff=cutoff, dtype=np.float32, workers=-1, # float32 is enough for 0-1 similarity, and faster than float64, workers=-1 uses all available CPU cores for parallel processing
        )
        for i, j in np.argwhere(sims >= cutoff):
            rows.append((idx1[i], idx2[j], year, float(sims[i, j]) * 100))

    pairs = pd.DataFrame(
        rows, columns=[f"{name1}_index", f"{name2}_index", "year", "similarity"]
    )
    pairs.insert(1, f"{name1}_title", df1.loc[pairs[f"{name1}_index"], f"{name1}_title"].values)
    pairs.insert(3, f"{name2}_title", df2.loc[pairs[f"{name2}_index"], f"{name2}_title"].values)
    return pairs


def ambiguous_matches(df1, name1, df2, name2, threshold=THRESHOLD):
    """
    Pairs where a row of df1 matches several rows of df2 (or vice versa).
    Columns n_<other>_per_<name> say how many candidates each row has.
    """
    pairs = all_candidate_pairs(df1, name1, df2, name2, threshold)
    i1, i2 = f"{name1}_index", f"{name2}_index" 

    c1 = f"n_{name2}_per_{name1}" # column name for the number of candidates in df2 for each row in df1
    c2 = f"n_{name1}_per_{name2}"
    pairs[c1] = pairs.groupby(i1)[i2].transform("nunique") # count the number of unique candidates in df2 for each row in df1
    pairs[c2] = pairs.groupby(i2)[i1].transform("nunique")

    amb = pairs[(pairs[c1] > 1) | (pairs[c2] > 1)] # keep only the pairs where a row in df1 matches several rows in df2 or vice versa
    return amb.sort_values(["year", f"{name1}_title", f"{name2}_title"]).reset_index(drop=True)


def duplicate_keys(df, name):
    """Rows inside ONE dataset sharing the same normalized title + year."""
    key = [f"{name}_title_normalized", f"{name}_year"]
    mask = df[f"{name}_title_normalized"].ne("") & df.duplicated(subset=key, keep=False)
    return df[mask].sort_values(key)


# ------------------------------------------------------------
# Master build
# ------------------------------------------------------------
def build_master(tmdb_df, numbers_df, rotten_df, threshold=THRESHOLD):
    tmdb = prepare(tmdb_df, "tmdb", year_col="release_year")
    numbers = prepare(numbers_df, "numbers")
    rotten = prepare(rotten_df, "rotten", year_col="release_date_theaters_year")

    # Step 1: TMDB outer Numbers
    master = outer_join(
        tmdb, numbers,
        "tmdb_title_normalized", "tmdb_year",
        "numbers_title_normalized", "numbers_year",
        score_col="score_tmdb_numbers", threshold=threshold,
    )

    # Key for step 2: whichever of TMDB / Numbers is available 
    master["_key_title"] = master["tmdb_title_normalized"].combine_first( 
        master["numbers_title_normalized"]
    )
    master["_key_year"] = master["tmdb_year"].combine_first(master["numbers_year"])

    # Step 2: (TMDB + Numbers) outer Rotten
    master = outer_join(
        master, rotten,
        "_key_title", "_key_year",
        "rotten_title_normalized", "rotten_year",
        score_col="score_master_rotten", threshold=threshold,
    )

    # Final unified identifiers
    master["title"] = (
        master["tmdb_title"]
        .combine_first(master["numbers_title"])
        .combine_first(master["rotten_title"])
    )
    master["year"] = (
        master["tmdb_year"]
        .combine_first(master["numbers_year"])
        .combine_first(master["rotten_year"])
    )
    master["title_normalized"] = (
        master["_key_title"].combine_first(master["rotten_title_normalized"])
    )

    master["in_tmdb"] = master["tmdb_title"].notna()
    master["in_numbers"] = master["numbers_title"].notna()
    master["in_rotten"] = master["rotten_title"].notna()
    master["n_sources"] = master[["in_tmdb", "in_numbers", "in_rotten"]].sum(axis=1)

    # Remove helper / redundant columns
    master = master.drop(
        columns=[
            "_key_title", "_key_year",
            "tmdb_title_normalized", "numbers_title_normalized", "rotten_title_normalized",
            "tmdb_year", "numbers_year", "rotten_year",
        ],
        errors="ignore",
    )

    # Column order: identifiers first
    front = [
        "title", "year", "title_normalized",
        "in_tmdb", "in_numbers", "in_rotten", "n_sources",
        "score_tmdb_numbers", "score_master_rotten",
    ]
    rest = [c for c in master.columns if c not in front]
    master = master[front + rest]

    master = master.sort_values(["year", "title"]).reset_index(drop=True)
    master.insert(0, "master_id", master.index)
    return master


if __name__ == "__main__":
    tmdb_df = pd.read_csv("../../data/processed/tmdb_clean.csv")
    numbers_df = pd.read_csv("../../data/processed/the_numbers_worldwide_2026_cleaned.csv")
    rotten_df = pd.read_csv(
        "../../data/processed/rotten_tomatoes_cleaned.csv",
        dtype={"aspect_ratio": "string"},
        low_memory=False,
    )

    master = build_master(tmdb_df, numbers_df, rotten_df, threshold=THRESHOLD)

    os.makedirs("../../data/merged", exist_ok=True)
    master.to_csv("../../data/merged/master_dataset.csv", index=False)

    print("Master shape:", master.shape)
    print("\nRows by source combination:")
    print(
        master.groupby(["in_tmdb", "in_numbers", "in_rotten"])
        .size()
        .rename("n_movies")
        .reset_index()
    )
    print("\nSaved to data/merged/master_dataset.csv")

    tmdb    = prepare(tmdb_df,    "tmdb",    year_col="release_year")
    numbers = prepare(numbers_df, "numbers")
    rotten  = prepare(rotten_df,  "rotten",  year_col="release_date_theaters_year")

    # Rows matching several rows on the other side (at 100%)
    amb = ambiguous_matches(tmdb, "tmdb", numbers, "numbers", threshold=100)
    print("Displaying ambiguous matches between TMDB and Numbers at 100% similarity:")
    display(amb)

    # Same for the other two pairs
    print("Displaying ambiguous matches between TMDB and Rotten Tomatoes at 100% similarity:")
    display(ambiguous_matches(tmdb, "tmdb", rotten, "rotten"))
    print("Displaying ambiguous matches between The Numbers and Rotten Tomatoes at 100% similarity:")
    display(ambiguous_matches(numbers, "numbers", rotten, "rotten"))

    # Duplicates inside a single dataset (usually the root cause)
    print("Displaying duplicate keys inside TMDB:")
    print(len(duplicate_keys(tmdb, "tmdb")))
    print("Displaying duplicate keys inside Rotten Tomatoes:")
    print(len(duplicate_keys(rotten, "rotten")))
    print("Displaying duplicate keys inside The Numbers:")
    print(len(duplicate_keys(numbers, "numbers")))

Master shape: (81076, 36)

Rows by source combination:
   in_tmdb  in_numbers  in_rotten  n_movies
0    False       False       True      1635
1    False        True      False       193
2    False        True       True         4
3     True       False      False     77156
4     True       False       True      1485
5     True        True      False       416
6     True        True       True       187

Saved to data/merged/master_dataset.csv
Displaying ambiguous matches between TMDB and Numbers at 100% similarity:


,tmdb_index,tmdb_title,numbers_index,numbers_title,year,similarity,n_numbers_per_tmdb,n_tmdb_per_numbers
0,52,Glass,34,Glass,2019,100.0,1,2
1,3907,Glass,34,Glass,2019,100.0,1,2
2,9944,Monster Hunter,135,Monster Hunter,2020,100.0,1,2
3,19036,Monster Hunter,135,Monster Hunter,2020,100.0,1,2
4,9930,The Hunt,197,The Hunt,2020,100.0,1,4
5,11276,The Hunt,197,The Hunt,2020,100.0,1,4
6,17550,The Hunt,197,The Hunt,2020,100.0,1,4
7,18437,The Hunt,197,The Hunt,2020,100.0,1,4
8,9997,Unhinged,143,Unhinged,2020,100.0,1,2
9,12099,Unhinged,143,Unhinged,2020,100.0,1,2


Displaying ambiguous matches between TMDB and Rotten Tomatoes at 100% similarity:


,tmdb_index,tmdb_title,rotten_index,rotten_title,year,similarity,n_rotten_per_tmdb,n_tmdb_per_rotten
0,2660,Candy,1624,Candy,2019,100.0,1,2
1,6197,Candy,1624,Candy,2019,100.0,1,2
2,444,Driven,207,Driven,2019,100.0,1,3
3,1781,Driven,207,Driven,2019,100.0,1,3
4,5615,Driven,207,Driven,2019,100.0,1,3
...,...,...,...,...,...,...,...,...
76,63381,Revelations,2659,Revelations,2025,100.0,1,2
77,60463,Sovereign,1374,Sovereign,2025,100.0,1,2
78,62449,Sovereign,1374,Sovereign,2025,100.0,1,2
79,70338,The Musical,3122,The Musical,2026,100.0,1,2


Displaying ambiguous matches between The Numbers and Rotten Tomatoes at 100% similarity:


,numbers_index,numbers_title,rotten_index,rotten_title,year,similarity,n_rotten_per_numbers,n_numbers_per_rotten


Displaying duplicate keys inside TMDB:
4652
Displaying duplicate keys inside Rotten Tomatoes:
0
Displaying duplicate keys inside The Numbers:
0
